Title: tas_julia.ipynb

Purpose: This notebook is used to calculate the global warming level for the period of 2015-2024. It uses code from this notebook: https://everval.github.io/Global-Temperature-Anomalies/#introduction

Author: Onno Nennecke on 09.02.2026 Modified: 10.02.2026

Input data: 

- Next to the data downloaded inside of the notebook, Original Data can also be found here: https://www.metoffice.gov.uk/hadobs/hadcrut5/data/HadCRUT.5.1.0.0/download.html


In [1]:
# Install necessary packages
# This code installs the packages if they are not already installed.
using Pkg
Pkg.add(["Plots", "Dates", "CSV", "DataFrames", "HTTP"])

# Load necessary packages
using Pkg
Pkg.activate(pwd())
using Dates, CSV, DataFrames, HTTP, Statistics


   Resolving package versions...
     Project No packages added to or removed from `~/Code/Annual_Temperatures/Project.toml`
    Manifest No packages added to or removed from `~/Code/Annual_Temperatures/Manifest.toml`
  Activating project at `~/Code/Annual_Temperatures`


In [2]:
# Function to convert wide format data to long format

function longseries(data)
    height = size(data, 1) # Number of rows, equivalent to the number of years
    last_row = 12 - count(ismissing, data[end, 2:13]) # Number of non-missing months in the last year

    many = (height - 1) * 12 + last_row # Total number of months in the long format
    long = zeros(many, 1) # Long format array

    for ii = 1:(height-1) # Loop through all years except the last one
        for jj = 1:12 # Loop through all months
            long[(ii-1)*12+jj] = data[ii, jj+1]
        end
    end

    for jj = 1:last_row # Loop through the last year
        long[(height-1)*12+jj] = data[height, jj+1]
    end

    return long
end

longseries (generic function with 1 method)

## Hadcrut

In [3]:
# Download the HadCRUT temperature data 
# URL of the HadCRUT5 global monthly average CSV
hurl = "https://www.metoffice.gov.uk/hadobs/hadcrut5/data/HadCRUT.5.1.0.0/analysis/diagnostics/HadCRUT.5.1.0.0.analysis.summary_series.global.monthly.csv"

# Local filename to save
hfilename = "/climca/people/onennecke/HadCRUT5_global_monthly_average.csv"
open(hfilename, "w") do io
    write(io, HTTP.get(hurl).body)
end

rawhadcrut = CSV.read(hfilename, DataFrame);

rename!(rawhadcrut, :Time => :Date);
rename!(rawhadcrut, :"Anomaly (deg C)" => :RawTemperature);

hadcrut = rawhadcrut[!, [:Date, :RawTemperature]];

oldbase = mean(hadcrut[(hadcrut.Date.>=Date(1850, 1, 1)).&(hadcrut.Date.<Date(1900, 1, 1)), :RawTemperature]);
hadcrut[!, :Temp] = hadcrut[!, :RawTemperature] .- oldbase;

CSV.write(hfilename, hadcrut);

println(first(hadcrut, 5));

5×3 DataFrame
 Row │ Date        RawTemperature  Temp         
     │ Date        Float64         Float64      
─────┼──────────────────────────────────────────
   1 │ 1850-01-01       -0.733695  -0.37379
   2 │ 1850-02-01       -0.360436  -0.000531503
   3 │ 1850-03-01       -0.627131  -0.267227
   4 │ 1850-04-01       -0.605297  -0.245392
   5 │ 1850-05-01       -0.531517  -0.171612


In [4]:
# Calcualte mean temperature for period 1995-2014 and 2015-2024
mean_1995_2014 = mean(hadcrut[(hadcrut.Date.>=Date(1995, 1, 1)).&(hadcrut.Date.<Date(2015, 1, 1)), :Temp]);
mean_2015_2024 = mean(hadcrut[(hadcrut.Date.>=Date(2015, 1, 1)).&(hadcrut.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014, " °C");
println("Mean temperature anomaly (2015-2024): ", mean_2015_2024, " °C");

Mean temperature anomaly (1995-2014): 0.8679792232733333 °C
Mean temperature anomaly (2015-2024): 1.2583507013566666 °C


### GISTEMP

In [5]:
# Download the GISTEMP temperature data 
# URL of the GISTEMP global monthly average CSV
gurl = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts%2BdSST.csv"
# Local filename to save
gfilename = "/climca/people/onennecke/GISTEMP_global_monthly_average.csv"
# Download the file

open(gfilename, "w") do io
    write(io, HTTP.get(gurl).body)
end

longgistemp = CSV.read(gfilename, DataFrame, header=2, missingstring=["***"]);
gistemp = longseries(longgistemp)[:];
Tt = length(gistemp) - 1;

start = Date(1880, 1, 1); # Start date of the dataset
fin = start + Month(Tt); # End date of the dataset
fechas = collect(start:Month(1):fin); # Create a Date array

gistemp = DataFrame(:Date=>fechas, :RawTemp=>gistemp);

oldbase = mean(gistemp[(gistemp.Date.>=Date(1880, 1, 1)).&(gistemp.Date.<Date(1900, 1, 1)), :RawTemp]);
gistemp[!, :Temp] = gistemp[!, :RawTemp] .- oldbase .+ 0.038; # Adjust for pre-1880 data

CSV.write(gfilename, gistemp);

println(first(gistemp, 5));

5×3 DataFrame
 Row │ Date        RawTemp  Temp     
     │ Date        Float64  Float64  
─────┼───────────────────────────────
   1 │ 1880-01-01    -0.19  0.076625
   2 │ 1880-02-01    -0.25  0.016625
   3 │ 1880-03-01    -0.1   0.166625
   4 │ 1880-04-01    -0.17  0.096625
   5 │ 1880-05-01    -0.11  0.156625


In [6]:
# Calcualte mean temperature for period 1995-2014 and 2015-2024
mean_1995_2014_gt = mean(gistemp[(gistemp.Date.>=Date(1995, 1, 1)).&(gistemp.Date.<Date(2015, 1, 1)), :Temp]);
mean_2015_2024_gt = mean(gistemp[(gistemp.Date.>=Date(2015, 1, 1)).&(gistemp.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014_gt, " °C");
println("Mean temperature anomaly (2015-2024): ", mean_2015_2024_gt, " °C");

Mean temperature anomaly (1995-2014): 0.8437499999999999 °C
Mean temperature anomaly (2015-2024): 1.2534583333333331 °C


## NOAA

In [10]:
# Download the NOAAGlobalTemp temperature data 
# URL of the NOAAGlobalTemp global monthly average CSV ---yearmonth.asc
nurl = "https://www.ncei.noaa.gov/data/noaa-global-surface-temperature/v6/access/timeseries/aravg.mon.land_ocean.90S.90N.v6.0.0.202512.asc"
# Local filename to save
nfilename = "/climca/people/onennecke/NOAA_global_monthly_average.csv"

# Download the file
open(nfilename, "w") do io
    write(io, HTTP.get(nurl).body)
end

lines = readlines(nfilename)
cleaned_lines = [join(split(strip(line)), ",") for line in lines];

# Write to file
write(nfilename, join(cleaned_lines, "\n"));

rawnoaa = CSV.read(nfilename, DataFrame; delim=',', header=0);

fechas = Date.(rawnoaa.Column1, rawnoaa.Column2, 1); # Create Date column from Column1 and Column2
noaa = DataFrame(:Date=>fechas, :RawTemp=>rawnoaa.Column3);
oldbase = mean(noaa[(noaa.Date.>=Date(1850, 1, 1)).&(noaa.Date.<Date(1900, 1, 1)), :RawTemp]);
noaa[!, :Temp] = noaa[!, :RawTemp] .- oldbase;
CSV.write(nfilename, noaa);
println(first(noaa, 5));

5×3 DataFrame
 Row │ Date        RawTemp    Temp       
     │ Date        Float64    Float64    
─────┼───────────────────────────────────
   1 │ 1850-01-01  -0.769049  -0.292106
   2 │ 1850-02-01  -0.545582  -0.0686386
   3 │ 1850-03-01  -0.560153  -0.0832096
   4 │ 1850-04-01  -0.673599  -0.196656
   5 │ 1850-05-01  -0.603704  -0.126761


In [11]:
# Calcualte mean temperature for period 1995-2014 and 2015-2024
mean_1995_2014_noaa = mean(noaa[(noaa.Date.>=Date(1995, 1, 1)).&(noaa.Date.<Date(2015, 1, 1)), :Temp]);
mean_2015_2024_noaa = mean(noaa[(noaa.Date.>=Date(2015, 1, 1)).&(noaa.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014_noaa, " °C");
println("Mean temperature anomaly (2015-2024): ", mean_2015_2024_noaa, " °C");

Mean temperature anomaly (1995-2014): 0.7571490933333337 °C
Mean temperature anomaly (2015-2024): 1.1709760183333333 °C


### Berkeley Earth

In [12]:
# Download the Berkeley Earth temperature data
# URL of the Berkeley Earth global monthly average CSV
burl = "https://storage.googleapis.com/berkeley-earth-temperature-hr/global/Global_TAVG_monthly.txt"
# Local filename to save
bfilename = "/climca/people/onennecke/BerkeleyEarth_global_monthly_average.csv"
# Download the file
open(bfilename, "w") do io
    write(io, HTTP.get(burl).body)
end

rawtemp = CSV.read(bfilename, DataFrame, comment="%", delim=" ", ignorerepeated=true);


colnames = [:Year, :Month, :Anomaly_Monthly, :Unc_Monthly,
            :Anomaly_Annual, :Unc_Annual, :Anomaly_5yr, :Unc_5yr,
            :Anomaly_10yr, :Unc_10yr, :Anomaly_20yr, :Unc_20yr];
rename!(rawtemp, colnames);

rawtemp.Date = Date.(rawtemp.Year, rawtemp.Month, 1); # Create Date column from Year and Month
rename!(rawtemp, :Anomaly_Monthly => :RawTemperature);

berkeley = rawtemp[!, [:Date, :RawTemperature]];

oldbase = mean(rawtemp[(rawtemp.Date.>=Date(1850, 1, 1)).&(rawtemp.Date.<Date(1900, 1, 1)), :RawTemperature]);
rawtemp[!, :Temp] = rawtemp[!, :RawTemperature] .- oldbase;

berkeley.Temp = rawtemp.Temp;

CSV.write(bfilename, berkeley);

println(first(berkeley, 5));

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/onennecke/.julia/packages/CSV/XLcqT/src/file.jl:593


5×3 DataFrame
 Row │ Date        RawTemperature  Temp       
     │ Date        Float64         Float64    
─────┼────────────────────────────────────────
   1 │ 1850-01-01          -0.378  -0.120722
   2 │ 1850-02-01          -0.189   0.0682783
   3 │ 1850-03-01          -0.237   0.0202783
   4 │ 1850-04-01          -0.734  -0.476722
   5 │ 1850-05-01          -0.305  -0.0477217


In [13]:
# Calcualte mean temperature for period 1995-2014 and 2015-2024
mean_1995_2014_berkeley = mean(berkeley[(berkeley.Date.>=Date(1995, 1, 1)).&(berkeley.Date.<Date(2015, 1, 1)), :Temp]);
mean_2015_2024_berkeley = mean(berkeley[(berkeley.Date.>=Date(2015, 1, 1)).&(berkeley.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014_berkeley, " °C");
println("Mean temperature anomaly (2015-2024): ", mean_2015_2024_berkeley, " °C");

Mean temperature anomaly (1995-2014): 0.8573741666666665 °C
Mean temperature anomaly (2015-2024): 1.255053333333333 °C


## Overview

In [15]:
# Print all temperature anomalies together
println("Mean temperature anomalies (1995-2014):");
println("HadCRUT: ", mean_1995_2014, " °C");
println("GISTEMP: ", mean_1995_2014_gt, " °C");
println("NOAA: ", mean_1995_2014_noaa, " °C");
println("Berkeley Earth: ", mean_1995_2014_berkeley, " °C");
println("\nMean temperature anomalies (2015-2024):");
println("HadCRUT: ", mean_2015_2024, " °C");
println("GISTEMP: ", mean_2015_2024_gt, " °C");
println("NOAA: ", mean_2015_2024_noaa, " °C");
println("Berkeley Earth: ", mean_2015_2024_berkeley, " °C");

Mean temperature anomalies (1995-2014):
HadCRUT: 0.8679792232733333 °C
GISTEMP: 0.8437499999999999 °C
NOAA: 0.7571490933333337 °C
Berkeley Earth: 0.8573741666666665 °C

Mean temperature anomalies (2015-2024):
HadCRUT: 1.2583507013566666 °C
GISTEMP: 1.2534583333333331 °C
NOAA: 1.1709760183333333 °C
Berkeley Earth: 1.255053333333333 °C
